In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import gradio as gr
import os

In [ ]:
# 1. CONFIGURACIÓN DEL MODELO GEMMA 4
# Asegúrate de que tu NVIDIA_API_KEY esté en el docker-compose.yaml
instruct_llm = ChatNVIDIA(model="google/gemma-4-31b-it")

In [ ]:
# 2. DEFINICIÓN DE LAS CADENAS (LCEL)
# Cadena 1: Genera el poema inicial
prompt1 = ChatPromptTemplate.from_messages([
    ("user", "INSTRUCTION: Only respond in rhymes\n\nPROMPT: {input}")
])

In [4]:
# Cadena 2:
prompt2 = ChatPromptTemplate.from_messages([
    ("user", (
        "INSTRUCTION: Only responding in rhyme, change the topic of the input poem to be about {topic}!"
        " Make it happy! Try to keep the same sentence structure, but make sure it's easy to recite!"
        "\n\nOriginal Poem: {input}\n\nNew Topic: {topic}"
    ))
])

In [5]:
# Construcción de cadenas con el operador pipe |
chain1 = prompt1 | instruct_llm | StrOutputParser()
chain2 = prompt2 | instruct_llm | StrOutputParser()

In [6]:
# 3. IMPLEMENTACIÓN DE LA LÓGICA DEL AGENTE 
def rhyme_chat2_stream(message, history, return_buffer=True):
    first_poem = None
    for entry in history:
        if entry.get("role") == "assistant":
            content = entry.get("content", "")
            if "Let me think!" in content:
                parts = content.split("\n\n")
                if len(parts) > 2:
                    first_poem = parts[1]
                    break

    if first_poem is None:
        # CASO A: Generación inicial
        buffer = "¡Oh! ¡Puedo hacer un poema maravilloso sobre eso! Let me think!\n\n"
        yield buffer if return_buffer else buffer

        for token in chain1.stream({"input": message}):
            buffer += token
            yield buffer if return_buffer else token

        passage = "\n\n¡Ahora déjame reescribirlo con un enfoque diferente! ¿Cuál debería ser el nuevo tema?"
        buffer += passage
        yield buffer if return_buffer else passage

    else:
        buffer = f"¡Seguro! Aquí tienes una versión sobre {message}:\n\n"
        yield buffer if return_buffer else buffer
        
        for token in chain2.stream({"input": first_poem, "topic": message}):
            buffer += token
            yield buffer if return_buffer else token

        passage = "\n\n¡Esto es divertido! ¡Dame otro tema!"
        buffer += passage
        yield buffer if return_buffer else passage

In [7]:
# 4. LANZAMIENTO DE LA INTERFAZ 
chatbot = gr.Chatbot(value=[{"role": "assistant", "content": "¡Te ayudaré a crear un poema! ¿Sobre qué quieres que escriba?"}])
demo = gr.ChatInterface(rhyme_chat2_stream, chatbot=chatbot).queue()

In [ ]:
demo.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://08aae042b2adb2938c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
